# Demo

In [14]:
import wampy as wam
import wampy.utils.debug as wam_debug

In [17]:
src = r"""
male(anakin).
male(luke).
"""

ast_program, symbol_table = wam.parse(src)
compiled_program = wam.compile(ast_program)
wam.query_from_str(compiled_program, "male(X).", symbol_table)

['male(anakin).', 'male(luke).']

In [18]:
wam_debug.debug_compiled_program(compiled_program, symbol_table)


=== Predicate entry points ===
male/1 -> PC 0

=== WAM instructions ===
0000: TRY          3   0   0
0001: GET_CONST    0   0   0
0002: PROCEED      0   0   0
0003: TRUST        0   0   0
0004: GET_CONST    1   0   0
0005: PROCEED      0   0   0


### Using onto-compilation

In [ ]:
src = r"""
male(anakin).
parent(anakin, luke).
father(A, B):- parent(A, B), male(A).
"""
ast_program, symbol_table = wam.parse(src)

compile_program = wam.compile(ast_program)
entry = compile_program.entry.copy()
size = compile_program.pc.value


src_onto = r"""
parent(padme, luke).
"""
ast_program_onto, symbol_table = wam.parse_with_symbol_table(src_onto, symbol_table)

compiled_program_onto = wam.compile_program_onto(compile_program, entry, size, ast_program_onto)
wam.query_from_str(compiled_program_onto, "parent(X, luke)", symbol_table)

['parent(padme, luke).', 'parent(anakin, luke).']

In [11]:
# wam.debug_compiled_program(compiled_program)

### Using custom config

In [ ]:
# Config
WAM_CONFIG_TOML = """
[ast]
num_terms = 60
max_terms_nodes = 16
max_terms_nodes_childs = 3

[stack]
heap_size = 2_048
trail_size = 512
cp_size = 256
unify_stack_size = 256

[entry]
max_functors = 256
max_arity = 4

[solver]
answer_max_answers = 1
"""


config = wam.load_config_toml(WAM_CONFIG_TOML)

src = r"""
male(anakin).
male(luke).
"""

ast_program, symbol_table = wam.parse(src, config)
compiled_program = wam.compile(ast_program, None, config)
wam.query_from_str(compiled_program, "male(X)", symbol_table, config)

['male(anakin).']